# Delta Lake com Apache Spark (PySpark)

Demonstração de **DDL, INSERT, UPDATE e DELETE** em uma tabela Delta.

## Cenário

Tabela única **`carros`**: o cadastro de veículos de uma seguradora. O dataset (`data/carros.csv`, com 8 registros) foi criado pelo grupo a partir do exemplo visto em aula.

### Modelo ER

```mermaid
erDiagram
    CARROS {
        string placa PK
        string modelo
        string chassi
        string marca
        int ano
        string cor
    }
```

| Coluna | Tipo | Descrição |
|---|---|---|
| placa | STRING | Placa do veículo (chave) |
| modelo | STRING | Modelo |
| chassi | STRING | Número do chassi |
| marca | STRING | Fabricante |
| ano | INT | Ano de fabricação |
| cor | STRING | Cor |

## 1. SparkSession com Delta Lake

`configure_spark_with_delta_pip` adiciona automaticamente o JAR do Delta compatível com a versão instalada pelo pip.

In [ ]:
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.appName("delta-lake")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Pasta onde a tabela Delta será gravada (recriada a cada execução)
CAMINHO = (Path("..") / "warehouse" / "delta" / "carros").resolve()
spark.sql("DROP TABLE IF EXISTS carros")
if CAMINHO.exists():
    import shutil

    shutil.rmtree(CAMINHO)

spark.version

## Dados de origem (CSV)

In [ ]:
df = spark.read.csv(
    "../data/carros.csv",
    header=True,
    schema="placa STRING, modelo STRING, chassi STRING, marca STRING, ano INT, cor STRING",
)
df.createOrReplaceTempView("carros_csv")
df.show()

## 2. DDL: criação da tabela Delta

In [ ]:
spark.sql(f"""
    CREATE TABLE carros (
        placa  STRING,
        modelo STRING,
        chassi STRING,
        marca  STRING,
        ano    INT,
        cor    STRING
    )
    USING delta
    LOCATION '{CAMINHO}'
""")

spark.sql("INSERT INTO carros SELECT * FROM carros_csv")  # carga inicial
spark.sql("SELECT * FROM carros ORDER BY placa").show()

## 3. INSERT

Inclusão de um carro novo.

In [ ]:
spark.sql("""
    INSERT INTO carros VALUES ('KLM2468', 'ONIX', '11223344556', 'GM', 2023, 'PRATA')
""")
spark.sql("SELECT * FROM carros ORDER BY placa").show()

## 4. UPDATE

O ECO SPORT (placa `EEE1056`) foi pintado de vermelho.

In [ ]:
spark.sql("UPDATE carros SET cor = 'VERMELHO' WHERE placa = 'EEE1056'")
spark.sql("SELECT * FROM carros WHERE placa = 'EEE1056'").show()

## 5. DELETE

Exclusão dos carros fabricados antes de 2015 (CLIO 2011 e PUNTO 2013).

In [ ]:
spark.sql("DELETE FROM carros WHERE ano < 2015")
spark.sql("SELECT * FROM carros ORDER BY placa").show()

## 6. Histórico e time travel

Cada operação gera uma nova **versão** da tabela no `_delta_log`.

In [ ]:
spark.sql("DESCRIBE HISTORY carros").select("version", "timestamp", "operation").show(truncate=False)

Consultando a tabela como ela estava **logo após a carga inicial** (versão 1):

In [ ]:
spark.sql("SELECT * FROM carros VERSION AS OF 1 ORDER BY placa").show()

## 7. Arquivos gerados

Dados em **Parquet** + log de transações em **JSON** (`_delta_log`).

In [ ]:
for arquivo in sorted(CAMINHO.rglob("*")):
    if arquivo.is_file() and not arquivo.name.endswith(".crc"):
        print(arquivo.relative_to(CAMINHO))

In [ ]:
spark.stop()